# BigQuery Hands-On: From Zero to Petabyte Scale

This notebook walks you through BigQuery using real public datasets. By the end, you'll have queried billions of rows, joined multiple datasets, run analytical functions, and trained a machine learning model — all using SQL.

**Datasets we'll use:**
- `bigquery-public-data.github_repos` — the GitHub Archive (billions of commits, files, and contents)
- `bigquery-public-data.stackoverflow` — Stack Overflow questions, answers, and tags
- `bigquery-public-data.usa_names` — warm-up dataset to get oriented

**Before each query**, look at the "This query will process X" estimate in the BigQuery UI (or check `dry_run` in code). Get in the habit of knowing the cost before you run.

---

## Setup

Install dependencies (skip if already installed) and authenticate.

In [ ]:
# Install the client library and pandas integration
!pip install --quiet google-cloud-bigquery google-cloud-bigquery-storage db-dtypes pandas matplotlib

In [ ]:
# Authenticate. Pick the option matching your environment:

# Option A — Local Jupyter (uses gcloud CLI credentials):
# Run this in a terminal first:  gcloud auth application-default login

# Option B — Google Colab:
from google.colab import auth
auth.authenticate_user()

# Option C — Service account JSON key:
# import os
# os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '/path/to/key.json'

PROJECT_ID = 'bigdata-course-495711'  # <-- replace with your GCP project ID

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)
print(f'Connected to project: {client.project}')

In [ ]:
# Load the BigQuery cell magic — lets us write SQL directly in cells
%load_ext google.cloud.bigquery

import pandas as pd
import matplotlib.pyplot as plt

---
## Step 1 — Warm-up: Your first query

Let's start small. The `usa_names` dataset has every baby name registered with the SSA from 1910 to today. Tiny by BigQuery standards — about 6 million rows.

**Goal:** Get oriented with the syntax. Notice the three-part table reference: `` `project.dataset.table` ``.

In [ ]:
%%bigquery top_names
SELECT
  name,
  SUM(number) AS total_babies
FROM `bigquery-public-data.usa_names.usa_1910_current`
WHERE gender = 'F'
GROUP BY name
ORDER BY total_babies DESC
LIMIT 10

In [ ]:
top_names

**What just happened?** The `%%bigquery top_names` magic ran the SQL and saved the result as a pandas DataFrame called `top_names`. You can now use it like any other DataFrame.

**Try it yourself:** Modify the query above to find the top 10 male names instead.

---
## Step 2 — Querying GitHub at scale

Now for something you couldn't do on your laptop. The `github_repos.commits` table contains **over 250 million commits** from millions of public repositories. We're going to find the most common commit messages across all of GitHub.

**Watch the bytes processed indicator** — this query will scan tens of GBs in seconds.

> ⚠️ **Cost note:** This query processes ~25 GB. The free tier allows 1 TB/month, so you can run this ~40 times for free. We'll select only the columns we need to keep costs down.

In [ ]:
%%bigquery common_commits
SELECT
  TRIM(LOWER(SPLIT(message, '\n')[OFFSET(0)])) AS first_line,
  COUNT(*) AS occurrences
FROM `bigquery-public-data.github_repos.commits`
WHERE message IS NOT NULL
  AND LENGTH(message) BETWEEN 5 AND 100
GROUP BY first_line
ORDER BY occurrences DESC
LIMIT 20

In [ ]:
common_commits

Take a moment to appreciate what just happened. You scanned **hundreds of millions of commits** in a few seconds — no cluster setup, no Spark configuration, no waiting in a queue.

**Try it yourself:** Modify the query to find the most common commit messages that are *exactly* `"fix typo"` (case-insensitive). How many people have made that exact commit?

In [ ]:
# Quick visualization
common_commits.head(15).plot(
    kind='barh',
    x='first_line',
    y='occurrences',
    figsize=(10, 6),
    legend=False
)
plt.title('Most Common Commit Messages on GitHub')
plt.xlabel('Occurrences')
plt.ylabel('')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

---
## Step 3 — Aggregations and filtering: Programming language popularity

The `github_repos.languages` table tells us which programming languages each repo uses (and how many bytes of code in each). Let's find the most-used languages on GitHub by total bytes.

In [ ]:
%%bigquery lang_popularity
SELECT
  lang.name AS language,
  COUNT(DISTINCT repo_name) AS repo_count,
  ROUND(SUM(lang.bytes) / POW(1024, 4), 2) AS total_terabytes
FROM `bigquery-public-data.github_repos.languages`,
     UNNEST(language) AS lang
GROUP BY language
ORDER BY total_terabytes DESC
LIMIT 20

In [ ]:
lang_popularity

**New concept: `UNNEST`.** BigQuery supports nested/repeated fields (arrays inside rows). Each repo has an *array* of language records. `UNNEST` flattens that array so we can aggregate across all of them. This nested structure is one of BigQuery's most powerful features for semi-structured data.

**Discussion question:** Is "total bytes" a fair measure of language popularity? What other metrics might you use, and how would the ranking change?

---
## Step 4 — Joining across datasets: GitHub meets Stack Overflow

Now let's combine two completely separate datasets. We'll compare:
- How popular each language is on **GitHub** (by repo count)
- How much it's discussed on **Stack Overflow** (by question count)

This kind of cross-dataset analysis is where BigQuery really shines — both datasets sit in the same warehouse and a single query can span them.

In [ ]:
%%bigquery github_vs_so
WITH github_langs AS (
  SELECT
    LOWER(lang.name) AS language,
    COUNT(DISTINCT repo_name) AS github_repos
  FROM `bigquery-public-data.github_repos.languages`,
       UNNEST(language) AS lang
  GROUP BY language
),
so_langs AS (
  SELECT
    tag AS language,
    COUNT(*) AS so_questions
  FROM `bigquery-public-data.stackoverflow.posts_questions`,
       UNNEST(SPLIT(tags, '|')) AS tag
  WHERE tag IN ('python','javascript','java','c++','go','rust','ruby',
                'php','typescript','c#','swift','kotlin','scala','r')
  GROUP BY tag
)
SELECT
  g.language,
  g.github_repos,
  s.so_questions,
  ROUND(s.so_questions / g.github_repos, 2) AS questions_per_repo
FROM github_langs g
JOIN so_langs s USING (language)
ORDER BY questions_per_repo DESC

In [ ]:
github_vs_so

**New concepts:**
- **CTEs (`WITH ... AS`)** — let you build queries from named subqueries. Much more readable than nested subqueries.
- **Cross-dataset joins** — note we joined `github_repos` and `stackoverflow`, two unrelated datasets, in a single query.

**Discussion:** A high `questions_per_repo` ratio means people ask a lot of questions relative to how much code exists. What might that say about a language's learning curve, ecosystem maturity, or community?

---
## Step 5 — Window functions: Ranking commits by author over time

Window functions perform calculations across rows *related to* the current row, without collapsing them. This is essential for analytical work — running totals, rankings, moving averages, year-over-year comparisons.

Let's find the top 3 most prolific commit authors per year for the past several years.

In [ ]:
%%bigquery top_authors_yearly
WITH yearly_commits AS (
  SELECT
    EXTRACT(YEAR FROM author.date) AS year,
    author.email AS author_email,
    COUNT(*) AS commit_count
  FROM `bigquery-public-data.github_repos.commits`
  WHERE EXTRACT(YEAR FROM author.date) BETWEEN 2018 AND 2022
    AND author.email IS NOT NULL
    AND author.email NOT LIKE '%noreply%'
  GROUP BY year, author_email
),
ranked AS (
  SELECT
    year,
    author_email,
    commit_count,
    RANK() OVER (PARTITION BY year ORDER BY commit_count DESC) AS rank_in_year
  FROM yearly_commits
)
SELECT year, rank_in_year, author_email, commit_count
FROM ranked
WHERE rank_in_year <= 3
ORDER BY year DESC, rank_in_year

In [ ]:
top_authors_yearly

**The window function pattern:**
```
RANK() OVER (PARTITION BY year ORDER BY commit_count DESC)
```
Reads as: "Rank rows, restarting the rank within each year, ordered by commit count descending."

Many of these top authors are bots (Dependabot, renovate, etc.). That's a lesson in itself: real-world data is messy, and a huge fraction of GitHub activity is automated.

**Try it yourself:** Modify the query to *exclude* obvious bots (look for emails containing 'bot', 'noreply', '[bot]').

---
## Step 6 — BigQuery ML: Train a model with SQL

BigQuery ML lets you train machine learning models using only SQL — no exporting data, no separate ML pipeline. You can build linear regression, logistic regression, k-means, time series, deep neural nets, and even use pretrained models, all from inside BigQuery.

We'll build a logistic regression model to predict whether a Stack Overflow question will get an accepted answer, based on simple features like tag count, title length, and time of day posted.

> ⚠️ **Cost note:** Training models costs more than queries. The query below uses a small sample (10,000 rows) to keep it cheap and fast. You'll need a dataset in your project to store the model — replace `your_dataset` below.

In [ ]:
DATASET = 'your_dataset'  # <-- replace with a dataset in your project, or create one:
# client.create_dataset(DATASET, exists_ok=True)

In [ ]:
%%bigquery --params $params
CREATE OR REPLACE MODEL `{dataset}.answered_question_model`
OPTIONS(
  model_type = 'logistic_reg',
  input_label_cols = ['has_accepted_answer']
) AS
SELECT
  ARRAY_LENGTH(SPLIT(tags, '|')) AS num_tags,
  LENGTH(title) AS title_length,
  EXTRACT(HOUR FROM creation_date) AS hour_posted,
  EXTRACT(DAYOFWEEK FROM creation_date) AS day_of_week,
  score AS question_score,
  CASE WHEN accepted_answer_id IS NOT NULL THEN 1 ELSE 0 END AS has_accepted_answer
FROM `bigquery-public-data.stackoverflow.posts_questions`
WHERE creation_date BETWEEN '2020-01-01' AND '2020-12-31'
LIMIT 10000

*Note: the `%%bigquery` magic doesn't directly substitute `{dataset}` — for the demo, just paste your dataset name into the SQL above, or use the Python client below.*

In [ ]:
# Cleaner approach using the Python client with f-string substitution
train_sql = f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET}.answered_question_model`
OPTIONS(
  model_type = 'logistic_reg',
  input_label_cols = ['has_accepted_answer']
) AS
SELECT
  ARRAY_LENGTH(SPLIT(tags, '|')) AS num_tags,
  LENGTH(title) AS title_length,
  EXTRACT(HOUR FROM creation_date) AS hour_posted,
  EXTRACT(DAYOFWEEK FROM creation_date) AS day_of_week,
  score AS question_score,
  CASE WHEN accepted_answer_id IS NOT NULL THEN 1 ELSE 0 END AS has_accepted_answer
FROM `bigquery-public-data.stackoverflow.posts_questions`
WHERE creation_date BETWEEN '2020-01-01' AND '2020-12-31'
LIMIT 10000
"""

print('Training model... (this may take 1-2 minutes)')
client.query(train_sql).result()
print('Done!')

In [ ]:
# Evaluate the model
eval_sql = f"""
SELECT *
FROM ML.EVALUATE(MODEL `{PROJECT_ID}.{DATASET}.answered_question_model`)
"""

evaluation = client.query(eval_sql).to_dataframe()
evaluation

In [ ]:
# Inspect the feature weights
weights_sql = f"""
SELECT *
FROM ML.WEIGHTS(MODEL `{PROJECT_ID}.{DATASET}.answered_question_model`)
"""

weights = client.query(weights_sql).to_dataframe()
weights

**What you just did:** trained a logistic regression model, evaluated it, and inspected its feature weights — without ever leaving SQL or moving data anywhere. The model lives in your BigQuery dataset and can be used for prediction with `ML.PREDICT(...)`.

BigQuery ML supports many model types beyond this:
- Linear and logistic regression
- K-means clustering
- Matrix factorization (recommendation systems)
- Time series forecasting (ARIMA+)
- Boosted trees
- Deep neural networks
- AutoML
- Importing TensorFlow/ONNX models
- Calling pretrained Vertex AI models (including LLMs) directly from SQL

---
## Wrap-up: What you've learned

In one notebook you've:
1. Run basic SELECT/GROUP BY queries
2. Scanned hundreds of millions of GitHub commits in seconds
3. Worked with nested/repeated fields using `UNNEST`
4. Joined two completely separate public datasets
5. Used window functions for ranked analytics
6. Trained, evaluated, and inspected a machine learning model — all in SQL

**Key takeaways:**
- BigQuery removes the infrastructure barrier: no clusters, no provisioning, no waiting
- The cost model rewards good query hygiene — select only the columns you need, filter early, use partitioned tables when possible
- Storage and compute are separate, so the same data can be queried by many users without contention
- SQL goes a lot further than most people think — including into ML

**Where to go next:**
- Explore other public datasets at https://cloud.google.com/bigquery/public-data
- Try `EXPLAIN` to understand query execution plans
- Look into partitioned and clustered tables for cost optimization
- Try BigQuery's geospatial functions or its built-in vector search

**Exercises to try on your own:**
1. Find the most-starred repos by language using `github_repos.sample_repos`
2. Identify which Stack Overflow tags are growing fastest year-over-year
3. Build a k-means clustering model that groups Stack Overflow questions by features
4. Combine GitHub commits with the `bigquery-public-data.crypto_bitcoin` dataset to find any commits referencing Bitcoin addresses (just for fun)